# OVD-Diagnose — Domain 2: Medical (VinDr chest X-ray)

14 finding classes with boxes. Provides the cross-domain contrast plus the
**three-way localization check** (SAM / detector-intrinsic `L_det` / domain proposer),
bootstrap CIs, reliability diagrams, qualitative examples, and controls.

**Data:** Kaggle → Add Data → `awsaf49/vinbigdata-512-image-dataset`.
*(Optional but valuable: also add a VinBigData-trained YOLO weight for the domain proposer.)*

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())

In [ ]:
REPO = 'https://github.com/ShMazumder/YOLOBERT.git'
import os
if not os.path.isdir('/kaggle/working/YOLOBERT'):
    !git clone $REPO /kaggle/working/YOLOBERT
%cd /kaggle/working/YOLOBERT
!git pull -q || true
!pip install -q ultralytics transformers pycocotools pyyaml pillow sentence-transformers

## Data — VinDr → COCO (boxes auto-rescaled to the 512px PNGs)

In [ ]:
CSV  = '/kaggle/input/datasets/awsaf49/vinbigdata-512-image-dataset/vinbigdata/train.csv'
IMGS = '/kaggle/input/datasets/awsaf49/vinbigdata-512-image-dataset/vinbigdata/train'
OUT  = 'data/medical/annotations/instances_val.json'
import os
print('csv exists:', os.path.exists(CSV), '| imgs exists:', os.path.isdir(IMGS))

In [ ]:
!python tools/vindr_to_coco.py --csv "{CSV}" --imgs "{IMGS}" --out "{OUT}" --img_ext .png
from pycocotools.coco import COCO
c = COCO(OUT); bb = [a['bbox'] for a in c.loadAnns(c.getAnnIds())[:8]]
mx = max(v for b in bb for v in b)
print('max box coord:', round(mx), '-> OK (scaled)' if mx <= 600 else '-> UNSCALED: stop and fix')

## Run all models (`LIMIT=0` full 4394; `200` quick)

In [ ]:
LIMIT = 0
limit_arg = f'--limit {LIMIT}' if LIMIT else ''
!python tools/run_all.py \
  --ann  data/medical/annotations/instances_val.json \
  --imgs "{IMGS}" \
  --out  runs/diag/medical \
  --device cuda:0 {limit_arg} --bootstrap 1000 \
  --models "yoloworld:yolov8s-world.pt,owlv2:google/owlv2-base-patch16-ensemble,groundingdino:IDEA-Research/grounding-dino-tiny" \
  --sam_weights mobile_sam.pt

In [ ]:
import pandas as pd
print(pd.read_csv('runs/diag/medical/fingerprints.csv').to_string(index=False))

## Domain-proposer check — answers "SAM fails on X-rays, so L is meaningless"
Plug a chest-X-ray-trained detector in as a class-agnostic proposer. If it recovers findings,
low `AR_SAM` was SAM's fault; if it also fails, localization really is the wall.
Set `DOMAIN_W` to an added weight (cell self-skips if absent).

In [ ]:
DOMAIN_W = '/kaggle/input/CHANGE-ME/vindr_yolo.pt'   # <- medical detector weight
import os
if os.path.exists(DOMAIN_W):
    !python tools/run_agnostic.py --model proposer --weights "{DOMAIN_W}" \
      --ann data/medical/annotations/instances_val.json --imgs "{IMGS}" \
      --out runs/diag/medical_domainprop --limit 200 --device cuda:0
else:
    print('no domain weight at', DOMAIN_W, '- skipping; L stays SAM + L_det only')

In [ ]:
# Three-way localization comparison
import json, os
from tools.ovd_diagnose import diagnose
ann = 'data/medical/annotations/instances_val.json'
G = json.load(open('runs/diag/medical/owlv2/results_global.json'))
O = json.load(open('runs/diag/medical/owlv2/results_oracle.json'))
sam_p, dom_p = 'runs/diag/medical/results_agnostic.json', 'runs/diag/medical_domainprop/results_agnostic.json'
A_sam = json.load(open(sam_p)) if os.path.exists(sam_p) else None
print('L (SAM proposer)      :', round(diagnose(ann, G, O, results_agnostic=A_sam)['L'], 4))
print('L_det (detector-intr.):', round(diagnose(ann, G, O)['L_det'], 4))
if os.path.exists(dom_p):
    print('L (domain proposer)   :', round(diagnose(ann, G, O, results_agnostic=json.load(open(dom_p)))['L'], 4))

## Calibration + qualitative

In [ ]:
!python tools/plot_reliability.py \
  --ann data/medical/annotations/instances_val.json \
  --models yoloworld:runs/diag/medical/yoloworld/results_global.json \
           owlv2:runs/diag/medical/owlv2/results_global.json \
           groundingdino:runs/diag/medical/groundingdino/results_global.json \
  --out paper/figures/reliability_medical

In [ ]:
!python tools/qualitative.py \
  --ann data/medical/annotations/instances_val.json --imgs "{IMGS}" \
  --results runs/diag/medical/owlv2/results_global.json \
  --mode missed --n 6 --out paper/figures/qual_medical.png
from IPython.display import Image
Image('paper/figures/qual_medical.png')

## Validation controls (medical)
**Caveat:** medical AP$\approx$0 makes the temperature/S controls partly degenerate — the
controls are primarily demonstrated on aerial. Blur (the L axis) is meaningful here.

In [ ]:
!python tools/synthetic_controls.py --control temperature \
  --ann data/medical/annotations/instances_val.json \
  --results runs/diag/medical/owlv2/results_global.json \
  --out runs/controls/medical_owlv2_temperature.csv
!python tools/synthetic_controls.py --control blur \
  --ann data/medical/annotations/instances_val.json --imgs "{IMGS}" \
  --sam_weights mobile_sam.pt --out runs/controls/medical_blur.csv --limit 200 --device cuda:0

In [ ]:
import pandas as pd, os
for name in ['medical_owlv2_temperature', 'medical_blur']:
    p = f'runs/controls/{name}.csv'
    if os.path.exists(p):
        print(f'== {name} =='); print(pd.read_csv(p).to_string(index=False), '\n')

**Cross-domain read:** compare this `fingerprints.csv` with the aerial one. If `L` (SAM),
`L_det` (detector-intrinsic) and `L` (domain proposer) all sit near 1 on medical while aerial
retains measurable localizability, the localization conclusion is robust to proposer choice —
which is the central reviewer objection, answered empirically.